# Аналитика продаж магазина электроники

Этот notebook показывает полный учебный аналитический цикл на синтетическом наборе из **10 000 строк и 18 столбцов**. Одна строка описывает одну товарную позицию внутри заказа. Материал рассчитан на начинающих аналитиков: ключевые термины поясняются, а код снабжён русскими комментариями.

Логика работы: **бизнес-вопрос → загрузка и осмотр → качество данных → очистка/ETL → preprocessing → EDA → KPI → статистика → ABC-XYZ → RFM → подготовка данных для Yandex DataLens → выводы и ограничения**.

## 1. Цель анализа

Цель — понять структуру продаж учебного магазина электроники: динамику выручки и прибыли, вклад категорий/брендов/каналов, возвраты, скидки, товарный портфель и клиентские сегменты.

Результат предназначен для учебного руководителя розничной сети и служит примером того, как из «грязного» файла получить проверенные аналитические таблицы и BI-витрины.

## 2. Аналитические вопросы и гипотезы

Основные вопросы:

1. Как меняются выручка, прибыль и число заказов по месяцам?
2. Какие категории, бренды, города и каналы формируют основной результат?
3. Где выше скидки и возвраты?
4. Какие товары относятся к ABC-XYZ классам?
5. Какие клиентские сегменты выделяются по RFM?
6. Какие наблюдаемые различия подтверждаются статистическими тестами?

Рабочие гипотезы:

- **H1:** в онлайн-каналах скидка в среднем/по распределению выше, чем в офлайн-магазине;
- **H2:** доля возвратов различается между товарными категориями;
- **H3:** между размером скидки и количеством единиц в строке есть положительная монотонная связь.

Гипотеза — проверяемое предположение. Даже статистически значимый результат не доказывает причинность.

## 3. Данные, единица наблюдения и ожидаемый результат

Источник — синтетический CSV `data/raw/electronics_sales_10000.csv`.

**Grain / единица наблюдения:** одна товарная позиция внутри заказа. Поэтому `line_id` должен быть уникальным после удаления полных дублей, а `order_id` может повторяться.

Ожидаемый результат:

- очищенный line-level CSV;
- производные показатели выручки, прибыли и маржи;
- EDA и статистические проверки;
- ABC-XYZ по товарам;
- RFM по клиентам;
- отдельные CSV-витрины для Yandex DataLens;
- итоговые выводы, ограничения и рекомендации.

## 4. Подготовка окружения

`pandas` используется для таблиц, `NumPy` — для числовых расчётов, `matplotlib` — для графиков, `SciPy` — для статистических тестов. Пути задаются относительно корня проекта, чтобы notebook не зависел от компьютера автора.

In [ ]:
# Импортируем Path для кроссплатформенной работы с путями.
from pathlib import Path
# Импортируем warnings для управления некритичными предупреждениями.
import warnings
# Импортируем NumPy для числовых расчётов.
import numpy as np
# Импортируем pandas для работы с табличными данными.
import pandas as pd
# Импортируем matplotlib для построения графиков.
import matplotlib.pyplot as plt
# Импортируем статистические тесты из SciPy.
from scipy.stats import mannwhitneyu, chi2_contingency, spearmanr
# Импортируем display и Markdown для аккуратного вывода в notebook.
from IPython.display import display, Markdown

# Скрываем только FutureWarning, чтобы учебный вывод оставался читаемым.
warnings.filterwarnings('ignore', category=FutureWarning)
# Настраиваем читаемый стиль графиков.
plt.rcParams['axes.grid'] = True
# Устанавливаем шрифт с поддержкой кириллицы.
plt.rcParams['font.family'] = 'DejaVu Sans'
# Задаём размер графика по умолчанию.
plt.rcParams['figure.figsize'] = (11, 6)
# Показываем до 30 столбцов в DataFrame.
pd.set_option('display.max_columns', 30)
# Форматируем числа до двух знаков после запятой.
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

# Определяем корень проекта как текущую рабочую папку.
PROJECT_DIR = Path('.')
# Формируем путь к исходным данным.
RAW_PATH = PROJECT_DIR / 'data' / 'raw' / 'electronics_sales_10000.csv'
# Формируем путь к очищенным данным.
CLEAN_PATH = PROJECT_DIR / 'data' / 'processed' / 'electronics_sales_clean.csv'
# Формируем каталог BI-витрин.
DATALENS_DIR = PROJECT_DIR / 'data' / 'datalens'
# Формируем каталог для сохранения графиков.
CHART_DIR = PROJECT_DIR / 'outputs' / 'charts'
# Создаём каталоги, если они ещё не существуют.
DATALENS_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

## 5. Загрузка данных

`pd.read_csv()` читает CSV в `DataFrame`. Сразу проверяем, что файл существует и содержит ожидаемые **10 000 строк**.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 6. Словарь полей

Ключевой момент: `line_id` идентифицирует строку продажи, а `order_id` — заказ. Один заказ может содержать несколько товаров, поэтому для количества заказов нельзя использовать обычное число строк.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 7. Первичный осмотр данных

Проверяем типы, пропуски, описательную статистику и число уникальных значений. На этом этапе ничего не исправляем: сначала фиксируем симптомы качества.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 8. Проверка типов и преобразование даты

Дата в исходном файле намеренно записана в нескольких текстовых форматах. `format='mixed'` позволяет разобрать смешанные представления, а `errors='coerce'` делает нераспознанные даты явными пропусками.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 9. Пропущенные значения

Пропуск не равен нулю. Сначала измеряем количество и долю пропусков, затем применяем разные правила в зависимости от смысла поля.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 10. Дубликаты и уникальность идентификаторов

Полный дубль завышает строковые суммы. Проверяем и полные дубликаты, и повторяемость `line_id`. Повторяющийся `order_id` является нормой, потому что один заказ может включать несколько товарных позиций.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 11. Согласованность категориальных значений

Пробелы и регистр могут искусственно создать отдельные категории. Нормализуем строки через `strip()` и `casefold()`, затем возвращаем утверждённые русские названия.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 12. Числовые правила и поиск аномалий

Для учебного примера задаём проверяемые диапазоны:

- `quantity`: 1–10;
- `unit_price_rub`: 300–500 000 руб.;
- `discount_pct`: 0–0,60;
- `unit_cost_rub`: 100–400 000 руб.

Это **правила конкретного синтетического кейса**, а не универсальные границы для реального ритейла.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 13. Очистка данных (ETL)

Удаляем полные дубликаты, восстанавливаем пропуски и аномалии устойчивыми правилами. Для цен/себестоимости используем медиану по товару, для скидки — медиану по каналу и категории, для количества — медиану по категории. Категориальные пропуски восстанавливаем из устойчивого контекста.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 14. Контроль качества после очистки

Очистка завершена только после повторной проверки. `assert` делает требования исполняемыми: при нарушении notebook остановится до аналитических выводов.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 15. Preprocessing и производные признаки

Создаём признаки, необходимые для анализа и BI. Возврат в этом учебном кейсе упрощённо обнуляет выручку и себестоимость строки; это модельное допущение, а не бухгалтерское правило.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 16. Общие KPI продаж

Считаем показатели на правильной гранулярности. Количество заказов — это `nunique(order_id)`, а средний чек — общая чистая выручка / число уникальных заказов.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 17. Распределения и выбросы после очистки

После очистки выбросы могут оставаться как реальные редкие значения. Поэтому визуально проверяем распределения цены, выручки строки и скидки, не удаляя значения автоматически только из-за их редкости.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 18. Временная динамика

Агрегируем данные по месяцам: выручка, прибыль, уникальные заказы и возвраты. Для полного года это удобный базовый уровень анализа сезонности.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 19. Категории, бренды, каналы и города

Сравниваем не только выручку, но и прибыль, маржинальность и возвраты. Это защищает от вывода «лидер по выручке = лучший сегмент».

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 20. Топ товаров и брендов

Топ по выручке полезен, но далее дополняется ABC-XYZ: один рейтинг не показывает стабильность спроса.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 21. Возвраты и скидки

Возврат — риск для выручки и клиентского опыта. Скидка может стимулировать объём, но её эффект нельзя оценивать только по росту количества: важны прибыль и состав заказов.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 22. Корреляционный анализ

Используем коэффициент Спирмена для монотонных связей между числовыми признаками. Корреляция показывает связь, а не причинность.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 23. Статистическая проверка гипотез

Проверяем три гипотезы. Уровень значимости для учебного примера: `alpha = 0.05`.

- H1: Mann–Whitney U — сравнение распределений скидки у офлайн и онлайн строк;
- H2: χ² Пирсона — связь категории и факта возврата;
- H3: Spearman — монотонная связь скидки и количества.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 24. ABC-XYZ анализ товаров

### ABC

Товары сортируются по чистой выручке. Учебные пороги:

- A — до 80% накопленной выручки;
- B — следующие до 95%;
- C — оставшиеся.

### XYZ

Стабильность спроса оценивается по коэффициенту вариации месячного количества:

`CV = стандартное отклонение / среднее`.

Для этого синтетического кейса используются пороги **X ≤ 0,25; Y ≤ 0,50; Z > 0,50**. Это проектное учебное правило, а не универсальный стандарт.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 25. RFM-анализ клиентов

RFM использует три признака:

- **Recency** — сколько дней прошло с последней покупки клиента до контрольной даты;
- **Frequency** — сколько уникальных заказов сделал клиент;
- **Monetary** — сколько чистой выручки принесли его покупки.

Баллы 1–5 рассчитываются квантильным разбиением. Сегменты ниже — учебная схема, а не универсальная CRM-методология.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 26. Подготовка данных для Yandex DataLens

Создаём несколько витрин с **явной гранулярностью**. Это безопаснее, чем механически соединять продажи, RFM, ABC-XYZ и статистические результаты в одну широкую таблицу.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

## 27. Рекомендуемая структура дашборда DataLens

Основная вкладка строится на `datalens_sales_mart.csv`: KPI, месячная динамика, категории, каналы, города, товары и возвраты.

Отдельные вкладки используют `datalens_product_abc_xyz.csv`, `datalens_customer_rfm.csv` и `datalens_hypothesis_summary.csv`.

**Важно:** эти предрассчитанные таблицы не пересчитываются автоматически от селекторов основной line-level витрины. Подробная схема находится в `DATALENS_DASHBOARD_GUIDE.md`.

## 28. Выводы, ограничения и рекомендации

Финальный текст ниже формируется из выполненных расчётов. Мы разделяем измеренные факты, интерпретацию и рекомендации.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

### Финальный QA

Перед завершением проверяем существование ключевых файлов и согласованность основных размеров.

In [ ]:
# TODO: выполните расчёты этого шага самостоятельно.
# Подсказка: сверяйтесь с описанием над ячейкой, проверяйте промежуточный результат
# и не переходите к следующему шагу, пока текущая проверка не объяснена.

### Источники по инструментам и методам

- pandas: https://pandas.pydata.org/docs/
- SciPy statistics: https://docs.scipy.org/doc/scipy/reference/stats.html
- Yandex DataLens — подключение к файлу: https://yandex.cloud/ru/docs/datalens/operations/connection/create-file
- Yandex DataLens — вычисляемые поля: https://yandex.cloud/ru/docs/datalens/concepts/calculations/
- Yandex DataLens — селекторы: https://yandex.cloud/ru/docs/datalens/dashboard/selector
- Yandex DataLens — связи виджетов: https://yandex.cloud/ru/docs/datalens/dashboard/link

Пороговые правила ABC-XYZ и схема RFM в этом notebook являются учебными проектными решениями, а не универсальными отраслевыми стандартами.